# 05 · Live stress-test — 10 rounds, logged for analysis

A `TARGET` switch (cell 2) picks the race:
- **`'news'`** — competition 5, the live-web RAG pipeline (Guardian API + headless-Chromium bodies).
- **`'math'`** — competition 3, the `pipeline_maths` recipe from notebook 03 (adaptive `cot_v2` /
  `structured_enumeration_cot` routing, **no** retrieval, **no** calculator, `max_new_tokens=300`).

Plays the chosen competition **10 times back to back**, each round its own logged run
(`{target}_r01`..`{target}_r10`), then aggregates everything — per-round accuracy + reached level, and
**every wrong question with its diagnostics** (News: the retrieved evidence; Math: the reasoning chain,
which routing strategy fired, and whether it hit the token cap). The output dir keys off `TARGET`
(`experiments/news_test/` or `experiments/math_test/`).

> Leaderboard attempts are FREE (only the cumulative best counts), so re-running 10 rounds costs us
> nothing on the board. We still pause politely between games (the PDF asks: no rapid requests).
> The RAG-vs-noRAG ablation (last section) is **News-only**.

## 1 · Setup — clone/sync the repo, paths, the provided client

In [71]:
# Auto-reload edited src modules on every cell run -- so after a `git pull` the newest code lands without a
# manual importlib.reload or a restart. (Re-run the cell that USES the code, e.g. code-wire.)
# Colab's IPython ships an autoreload that does `from imp import reload`, and `imp` is GONE in Python 3.12 --
# so a tiny `imp` shim (reload only) we install first, then load the extension. BEST-EFFORT: any failure
# caught, so the cell never stalls (fall back: after a src pull, Runtime > Restart to pick changes up).
try:
    import sys as _sys, types as _types, importlib as _importlib
    if 'imp' not in _sys.modules:
        _imp = _types.ModuleType('imp')
        _imp.reload = _importlib.reload          # the one thing the old autoreload.py wants from `imp`.
        _sys.modules['imp'] = _imp
    _ip = get_ipython()
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
    print('autoreload: ON (src edits hot-reload on cell re-run)')
except Exception as _e:
    print(f'autoreload OFF ({type(_e).__name__}: {_e}) -- after a src pull, Runtime > Restart to pick changes up.')

import os, sys

REPO_URL = 'https://github.com/SleepyEveryD/NLP.git'
REPO_ROOT = '/content/NLP'
BRANCH = 'maths'
if not os.path.exists(REPO_ROOT):
  !git clone -b {BRANCH} {REPO_URL} {REPO_ROOT}
else:
  # Already cloned -> HARD-SYNC to the latest pushed branch (fetch + force-reset to origin/{BRANCH}).
  # Tracked files are overwritten to match remote; UNTRACKED run outputs are KEPT (experiments/runs/* is
  # gitignored). NOTE: `git pull` updates the FILES on disk -- it does NOT refresh THIS notebook's cells.
  !cd {REPO_ROOT} && git fetch -q origin && git checkout -q -f -B {BRANCH} origin/{BRANCH}

!cd {REPO_ROOT} && echo "on branch:" $(git rev-parse --abbrev-ref HEAD) "@" $(git --no-pager log -1 --oneline)

SRC = os.path.join(REPO_ROOT, 'src')
API_CLIENT = os.path.join(REPO_ROOT, 'NLP_assignment_api_client')
for p in (SRC, API_CLIENT):
  if p not in sys.path:
    sys.path.insert(0, p)
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)

from millionaire_client import MillionaireClient
print('millionaire_client, imported it is.')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
autoreload: ON (src edits hot-reload on cell re-run)
on branch: maths @ d70d844 MATHS_LIVE_POLICY 移除 DISCRETE_ENUMERATION → 落回 cot_v2,只留 INTERVAL_COUNTING + TEMPORAL_REASONING 走 structured。
Repo root: /content/NLP
millionaire_client, imported it is.


In [72]:
# The inference stack + the client's `requests`, install we do (light it stays). `-U` kept (Colab a stale
# bitsandbytes preinstalls); pandas/requests PINNED to Colab's versions (bare `-U` breaks google-colab/cudf).
!pip install -q -U 'transformers>=4.45.0' 'accelerate>=0.34.0' 'bitsandbytes>=0.46.1' sentencepiece einops pyyaml 'pandas==2.2.2' matplotlib 'requests==2.32.4'
print('Installed, the dependencies are.')

Installed, the dependencies are.


In [73]:
# Headless Chromium -- the live-NEWS body fetch it powers (configs/live.yaml: news_body_mode "browser").
# The relevance gate now ROUTES off-topic Guardian results here, so the browser matters MORE for News.
# Skip this only if you set retrieval.news_body_mode: "off".
!pip install -q playwright
!playwright install chromium
!playwright install-deps

# ARMED? a REAL launch the surest test is. NOT ready -> News falls back to HEADLINES only (crash-safe).
try:
    from playwright.sync_api import sync_playwright
    with sync_playwright() as _p:
        _b = _p.chromium.launch(headless=True); _b.close()
    print('headless Chromium: READY -- live-News body fetch armed.')
except Exception as _e:
    print(f'headless Chromium NOT ready ({type(_e).__name__}: {_e})')
    print('   -> News will use HEADLINES only. Re-run this cell, or set retrieval.news_body_mode: "off".')

Installing dependencies...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,630 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' do

## 2 · Config — pick the race (`TARGET`) + how many rounds

In [74]:
from config import RunConfig

config = RunConfig.from_yaml(os.path.join(REPO_ROOT, 'configs', 'live.yaml'))

# --- Pick which race to stress-test -------------------------------------------------------------
TARGET = 'math'            # 'news' (comp 5, live-web RAG) | 'math' (comp 3, cot_v2 routing, NO RAG/tools)
# ------------------------------------------------------------------------------------------------

COMP_IDS = {'news': 5, 'math': 3}   # the competition id each target plays (confirm in the post-login list).
COMP_ID  = COMP_IDS[TARGET]
NUM_ROUNDS = 10            # how many live games to play, back to back.
PAUSE_S    = 8.0          # polite gap between games (PDF: no rapid consecutive requests).

config.game.competition_id = COMP_ID
config.game.game_mode = 'text'

# The Guardian Open Platform key -- from a Colab secret (NEVER hardcoded). Only News uses it, but harmless
# to set always. With it, the Guardian fast body path is armed; the relevance gate keeps it ONLY on-topic.
try:
    from google.colab import userdata as _ud
    config.retrieval.guardian_api_key = _ud.get('guardian_key') or ''
except Exception:
    config.retrieval.guardian_api_key = config.retrieval.guardian_api_key or ''

print('TARGET:', TARGET, '| competition_id:', COMP_ID, f'({TARGET})')
print('rounds:', NUM_ROUNDS, '| pause between:', PAUSE_S, 's')
print('aim_seconds:', config.game.aim_seconds, '| model:', config.model.name, '|', config.model.quantization)
if TARGET == 'news':
    print('RAG:', 'ON' if config.retrieval.enabled else 'OFF', '| source:', config.retrieval.source,
          '| news_body_mode:', config.retrieval.news_body_mode, '| fetch_bodies:', config.retrieval.news_fetch_bodies)
    print('Guardian API:', 'KEY SET' if config.retrieval.guardian_api_key else 'no key -> News uses browser')
else:  # math
    print('Maths pipeline: adaptive cot_v2 / structured_enumeration_cot routing, NO retrieval, NO calculator,'
          ' max_new_tokens=300 (30s-wall safe).')

TARGET: math | competition_id: 3 (math)
rounds: 10 | pause between: 8.0 s
aim_seconds: 25.0 | model: Qwen/Qwen2.5-7B-Instruct | 4bit
Maths pipeline: adaptive cot_v2 / structured_enumeration_cot routing, NO retrieval, NO calculator, max_new_tokens=300 (30s-wall safe).


## 3 · Load + warm up the model

In [75]:
import time
from inference.engine import TransformersEngine

t0 = time.perf_counter()
if 'engine' not in globals():
      engine = TransformersEngine(model_name=config.model.name,
                                  quantization=config.model.quantization,
                                  dtype=config.model.dtype)
      engine.warmup()
else:
      print('engine 已在显存中,跳过加载。')
print(f'Model loaded in {time.perf_counter() - t0:.1f}s')

t0 = time.perf_counter()
engine.warmup()
print(f'Warmup in {time.perf_counter() - t0:.1f}s')

engine 已在显存中,跳过加载。
Model loaded in 0.0s
Warmup in 0.6s


## 4 · Wire the pipeline for `TARGET` + log in to the game

In [76]:
from classify.classifier import QuestionClassifier
from prompting.builder import PromptBuilder, RoutingPromptBuilder
from classify.reasoning_router import MATHS_LIVE_POLICY, MATHS_LIVE_FALLBACK
from agent.pipeline import QAPipeline
from tools import default_tools
from retrieval import build_retriever

if TARGET == 'news':
    # Phase 4 RAG: the routing retriever. For News (post-cutoff) `routed` sends questions to the live web
    # (Guardian API + relevance gate -> headless-Chromium on the gnews link); `needs_retrieval` gates it.
    retriever = build_retriever(config.retrieval)
    print('RAG:', (f'ON  source={config.retrieval.source}  top_k={config.retrieval.top_k}') if retriever else 'OFF')

    # Exactly the SHARED pipeline competition 5 uses in notebook 03 -- few_shot + RAG + the calculator no-op.
    pipeline = QAPipeline(
        engine=engine,
        prompt_builder=PromptBuilder(strategy=config.prompt_strategy),
        classifier=QuestionClassifier(),
        retriever=retriever,
        tools=default_tools(),
        latency_budget_s=config.latency_budget_s,
    )
    print('News pipeline wired:', config.prompt_strategy, '+ RAG (routed) + classifier-gated tools')

else:  # math -- EXACTLY the `pipeline_maths` recipe from notebook 03 (comp 3).
    # Adaptive routing: counting / temporal / discrete-enumeration -> structured_enumeration_cot; everything
    # else (arithmetic, logic, concept/stats) -> the cot_v2 fallback. NO retrieval (it only distracts on
    # Maths), NO calculator (at n=1 it clobbers the chain on numeric Qs), single-pass. max_new_tokens=300 is
    # the REAL 30s-wall guard: at 512 a structured chain ran ~32s and the server returned timed_out (level 0).
    pipeline = QAPipeline(
        engine=engine,
        prompt_builder=RoutingPromptBuilder(policy=MATHS_LIVE_POLICY, fallback_strategy=MATHS_LIVE_FALLBACK),
        classifier=QuestionClassifier(),
        retriever=None,
        tools=None,
        latency_budget_s=config.latency_budget_s,
        max_new_tokens=300,
    )
    print('Maths pipeline wired: ADAPTIVE (counting/temporal/enum -> structured_enumeration_cot; else -> cot_v2)'
          ' + 300 tokens + single-pass + NO retrieval + NO calculator')

# --- Log in to the real game ---
from google.colab import userdata
from game.client import GameClient

USERNAME = userdata.get('username')
PASSWORD = userdata.get('password')

game_client = GameClient()
game_client.login(USERNAME, PASSWORD)
print('Logged in as', USERNAME)

# The competitions + ids (safe -- starts no timer). Confirm the target's id here (News=5, Maths=3).
for c in game_client.list_competitions():
    print('  id=', c.id, '|', c.name, '| max_levels=', getattr(c, 'max_levels', '?'))

Maths pipeline wired: ADAPTIVE (counting/temporal/enum -> structured_enumeration_cot; else -> cot_v2) + 300 tokens + single-pass + NO retrieval + NO calculator
Logged in as runjie dai
  id= 0 | Entertainment | max_levels= 15
  id= 1 | Ancient History and Politics | max_levels= 15
  id= 2 | Science and Nature | max_levels= 15
  id= 3 | Maths | max_levels= 15
  id= 4 | Philosophy and Psychology | max_levels= 15
  id= 5 | News | max_levels= 15


## 5 · ▶ Play 10 live rounds  (each its own logged run)

Plays the `TARGET` game `NUM_ROUNDS` times. Each round writes its own run dir `{target}_r{NN}` (so no
round overwrites another — the LiveRunner truncates *within* a run_id). One round failing (a rate-limit, a
network blip) is caught and logged as a gap — the loop carries on.

In [77]:
import time
from evaluation.runner import run_session

LOG_ROOT = os.path.join(REPO_ROOT, 'experiments', 'runs')
round_runs = []   # [(round_no, run_path-or-None)]

for r in range(1, NUM_ROUNDS + 1):
    config.run_id = f'{TARGET}_r{r:02d}'
    print(f'\n===== ▶ ROUND {r}/{NUM_ROUNDS}  (run_id={config.run_id}) =====')
    try:
        path = run_session(pipeline, config, game_client=game_client, log_root=LOG_ROOT)
        round_runs.append((r, path))
        print('   round log:', path)
    except Exception as e:
        round_runs.append((r, None))
        print(f'   ⚠️ round {r} FAILED ({type(e).__name__}: {e}) -- logged as a gap, the loop continues.')
    if r < NUM_ROUNDS:
        time.sleep(PAUSE_S)   # polite gap between live games.

print('\nAll rounds done. Logged runs:', [p for _r, p in round_runs if p])


===== ▶ ROUND 1/10  (run_id=math_r01) =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=6643 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=6.9s (left was 29.912115)
[2] qid=6691 lvl=0 reached=1 -> A | correct=False | timed_out=False | latency=7.4s (left was 29.911349)
   round log: /content/NLP/experiments/runs/math_r01

===== ▶ ROUND 2/10  (run_id=math_r02) =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=6976 lvl=0 reached=0 -> B | correct=False | timed_out=False | latency=4.9s (left was 29.912145)
   round log: /content/NLP/experiments/runs/math_r02

===== ▶ ROUND 3/10  (run_id=math_r03) =====


...

[1] qid=6741 lvl=0 reached=0 -> C | correct=False | timed_out=False | latency=16.1s (left was 29.912452)
   round log: /content/NLP/experiments/runs/math_r08

===== ▶ ROUND 9/10  (run_id=math_r09) =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=6986 lvl=0 reached=None -> C | correct=True | timed_out=False | latency=6.2s (left was 29.912223)
[2] qid=6776 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=4.3s (left was 29.912434)
[3] qid=6713 lvl=0 reached=2 -> D | correct=False | timed_out=False | latency=3.2s (left was 29.912853)
   round log: /content/NLP/experiments/runs/math_r09

===== ▶ ROUND 10/10  (run_id=math_r10) =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=6695 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=7.7s (left was 29.913501)
[2] qid=6766 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=4.5s (left was 29.912414)
[3] qid=6917 lvl=0 reached=None -> D | correct=True | timed_out=False | latency=4.6s (left was 29.911948)
[4] qid=6839 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=7.5s (left was 29.912839)
[5] qid=6841 lvl=0 reached=4 -> B | correct=False | timed_out=False | latency=11.0s (left was 29.912233)
   round log: /content/NLP/experiments/runs/math_r10

All rounds done. Logged runs: ['/content/NLP/experiments/runs/math_r01', '/content/NLP/experiments/runs/math_r02', '/content/NLP/experiments/runs/math_r03', '/content/NLP/experiments/runs/math_r04', '/content/NLP/experiments/runs/math_r05', '/content/NLP/experiments/runs/math_r06', '/content/NLP/experiments/runs/math_r07', '/content/NLP/experiments/runs/math_r08', '/content/NLP/experiments/runs/math_r09', '/conten

## 6 · Analysis — per-round scores + every wrong question (with diagnostics)

Aggregates all rounds and **saves** the consolidated records to `experiments/{TARGET}_test/` so they ride
back to the repo. Three artifacts: a per-round summary, every question, and the wrong questions alone.
- **News**: wrong questions carry the retrieved evidence text (retrieval-miss vs grounding-miss diagnosis)
  and a retrieval source mix.
- **Math**: wrong questions carry the reasoning chain (`raw_output`), which routing strategy fired
  (`prompt_strategy`), and the output token count — so you can see truncation at the 300-token cap.

In [78]:
import json, collections
from pathlib import Path
import pandas as pd

OUT_DIR = Path(REPO_ROOT) / 'experiments' / f'{TARGET}_test'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def _read_round(path):
    """One round's records.jsonl -> list of dict rows ([] if missing/empty)."""
    if not path:
        return []
    p = Path(path) / 'records.jsonl'
    if not p.exists():
        return []
    return [json.loads(l) for l in p.read_text(encoding='utf-8').splitlines() if l.strip()]

def _sources(row):
    """The retrieval SOURCE mix for a row, from retrieved_snippets ('[theguardian.com#..] ' prefix)."""
    out = collections.Counter()
    for s in (row.get('retrieved_snippets') or []):
        if isinstance(s, str) and s.startswith('['):
            out[s[1:].split('#', 1)[0].split(']', 1)[0]] += 1
    return dict(out)

# --- Gather every round ---
summary_rows, all_q, wrong_q = [], [], []
for r, path in round_runs:
    rows = _read_round(path)
    graded = [x for x in rows if x.get('correct') is not None]
    n_correct = sum(1 for x in graded if x.get('correct') is True)
    reached = [x.get('reached_level') for x in rows if x.get('reached_level') is not None]
    summary_rows.append({
        'round': r,
        'answered': len(rows),
        'correct': n_correct,
        'graded': len(graded),
        'accuracy': (n_correct / len(graded)) if graded else float('nan'),
        'reached_level': max(reached) if reached else None,
    })
    for x in rows:
        rec = {'round': r, **x, 'sources': _sources(x)}
        all_q.append(rec)
        if x.get('correct') is False:
            wrong_q.append(rec)

# --- Per-round summary + overall ---
summary = pd.DataFrame(summary_rows)
print(f'PER-ROUND SUMMARY ({TARGET})')
print(summary.to_string(index=False))
tot_c = int(summary['correct'].sum()); tot_g = int(summary['graded'].sum())
lv = [s['reached_level'] for s in summary_rows if s['reached_level'] is not None]
print(f"\nOVERALL: {tot_c}/{tot_g} graded = {tot_c / tot_g:.1%}" if tot_g else '\nOVERALL: no graded answers')
if lv:
    print(f"reached_level over {len(lv)} rounds: min={min(lv)} max={max(lv)} mean={sum(lv)/len(lv):.1f} | {sorted(lv, reverse=True)}")

if TARGET == 'news':
    # --- Retrieval source mix across ALL questions (did the gate route to the browser? did docs land?) ---
    src_total = collections.Counter()
    for x in all_q:
        src_total.update(x['sources'])
    print('\nRETRIEVAL SOURCE MIX (doc count across all', len(all_q), 'questions):', dict(src_total))
    fired = sum(1 for x in all_q if x.get('retrieval_used'))
    print(f"retrieval fired on {fired}/{len(all_q)} questions")
else:  # math -- which routing strategy fired, and how close to the 300-token cap the chains ran.
    strat_mix = collections.Counter(x.get('prompt_strategy') or '?' for x in all_q)
    print('\nROUTING STRATEGY MIX (across all', len(all_q), 'questions):', dict(strat_mix))
    toks = [x.get('tokens_out', 0) for x in all_q if x.get('tokens_out')]
    if toks:
        capped = sum(1 for t in toks if t >= 300)
        print(f"tokens_out: min={min(toks)} max={max(toks)} mean={sum(toks)/len(toks):.0f}"
              f" | hit the 300 cap on {capped}/{len(toks)} questions (likely truncated before 'Answer:')")

PER-ROUND SUMMARY (math)
 round  answered  correct  graded  accuracy  reached_level
     1         2        1       2  0.500000              1
     2         1        0       1  0.000000              0
     3         2        1       2  0.500000              1
     4         6        5       6  0.833333              5
     5         4        3       4  0.750000              3
     6         1        0       1  0.000000              0
     7         1        0       1  0.000000              0
     8         1        0       1  0.000000              0
     9         3        2       3  0.666667              2
    10         5        4       5  0.800000              4

OVERALL: 16/26 graded = 61.5%
reached_level over 10 rounds: min=0 max=5 mean=1.6 | [5, 4, 3, 2, 1, 1, 0, 0, 0, 0]

ROUTING STRATEGY MIX (across all 26 questions): {'cot_v2': 23, 'structured_enumeration_cot': 3}
tokens_out: min=12 max=160 mean=62 | hit the 300 cap on 0/26 questions (likely truncated before 'Answer:')


In [79]:
# --- Every WRONG question, with the right diagnostics for the target ---
#   News: the retrieved EVIDENCE (retrieval-miss vs grounding-miss check).
#   Math: the reasoning chain (raw_output), the routing strategy that fired, and tokens_out (cap = 300).
print(f"{'=' * 78}\nEVERY WRONG QUESTION  ({len(wrong_q)} across {NUM_ROUNDS} rounds)\n{'=' * 78}")
for x in wrong_q:
    opts = x.get('options') or {}
    pick = x.get('predicted_answer')
    if TARGET == 'news':
        print(f"\n[round {x['round']}] qid={x['qid']} reached_level={x.get('reached_level')} "
              f"lat={x.get('latency_s', 0):.1f}s sources={x['sources']}")
    else:
        toks = x.get('tokens_out', 0)
        print(f"\n[round {x['round']}] qid={x['qid']} level={x.get('level')} reached_level={x.get('reached_level')} "
              f"lat={x.get('latency_s', 0):.1f}s strategy={x.get('prompt_strategy')} "
              f"tokens_out={toks}{'  <-- HIT 300 CAP (truncated?)' if toks and toks >= 300 else ''}")
    print(f"Q: {x['question_text']}")
    for k, v in opts.items():
        print(f"   {k}. {v}" + ('  <-- our pick (WRONG)' if k == pick else ''))
    if TARGET == 'news':
        snips = x.get('retrieved_snippets') or []
        if snips:
            print('   -- retrieved evidence --')
            for s in snips:
                print(f"      {str(s)[:500]}")
        else:
            print('   (no retrieved evidence logged)')
    else:  # math -- the model's reasoning chain (truncation / wrong set-up shows here).
        raw = (x.get('raw_output') or '').strip()
        print('   -- reasoning chain --')
        print('      ' + (raw[:900].replace('\n', '\n      ') if raw else '(empty)'))

# --- SAVE the consolidated artifacts (ride back to the repo via experiments/) ---
summary.to_csv(OUT_DIR / 'summary.csv', index=False)
with open(OUT_DIR / 'all_questions.jsonl', 'w', encoding='utf-8') as f:
    for x in all_q:
        f.write(json.dumps(x, ensure_ascii=False) + '\n')
with open(OUT_DIR / 'wrong_questions.jsonl', 'w', encoding='utf-8') as f:
    for x in wrong_q:
        f.write(json.dumps(x, ensure_ascii=False) + '\n')
print(f"\nSaved -> {OUT_DIR}/  (summary.csv, all_questions.jsonl [{len(all_q)}], wrong_questions.jsonl [{len(wrong_q)}])")

EVERY WRONG QUESTION  (10 across 10 rounds)

[round 1] qid=6691 level=0 reached_level=1 lat=7.4s strategy=structured_enumeration_cot tokens_out=53
Q: A reading specialist in a large public school system believes that the more time students spend reading, the better they will do in school. She plans a middle school experiment in which an SRS of 30 eighth graders will be assigned four extra hours of reading per week, an SRS of 30 seventh graders will be assigned two extra hours of reading per week, and an SRS of 30 sixth graders with no extra assigned reading will be a control group. After one school year, the mean GPAs from each group will be compared. Is this a good experimental design?
   A. No, because while this design may point out an association between reading and GPA, it cannot establish a cause-and-effect relationship.  <-- our pick (WRONG)
   B. Yes.
   C. No, because grade level is a lurking variable which may well be confounded with the variables under consideration.
   D. N

## 8 · RAG vs no-RAG ablation — News-only — do knowledge questions answer better WITHOUT retrieval?

**Runs only when `TARGET == 'news'`** (Maths uses no retrieval, so there is nothing to ablate).

Some News questions are really **knowledge/historical** ("which US president visited China in 2008..") —
the model may know the answer from its **own training**, and an off-topic retrieved article can *mislead*
it (grounding on junk). This re-answers the SAME questions **with** retrieval and **without**, side by
side, so we can see where RAG helps vs hurts. Annotate `KNOWN_GOLD` for questions you can verify by hand
to get a score (live games hide the gold).

In [80]:
if TARGET != 'news':
    print(f"RAG-vs-noRAG ablation is News-only (Maths uses no retrieval) -- skipping for TARGET = {TARGET}.")
else:
    # Twin of the News pipeline but retriever=None -> pure parametric knowledge (no RAG).
    import json
    from schemas import Question, QuestionType
    from agent.pipeline import QAPipeline
    from prompting.builder import PromptBuilder
    from classify.classifier import QuestionClassifier
    from tools import default_tools

    pipeline_norag = QAPipeline(
        engine=engine,
        prompt_builder=PromptBuilder(strategy=config.prompt_strategy),
        classifier=QuestionClassifier(),
        retriever=None,                 # <- the only difference from the RAG `pipeline`.
        tools=default_tools(),
        latency_budget_s=config.latency_budget_s,
    )

    # Questions to probe. Default: the wrong questions THIS News run logged. Point SRC elsewhere to test others.
    SRC = os.path.join(REPO_ROOT, 'experiments', 'news_test', 'wrong_questions.jsonl')
    rows = [json.loads(l) for l in open(SRC, encoding='utf-8') if l.strip()]

    # Known gold BY TEXT substring (resolved to the option letter at runtime -> robust to option shuffling).
    # Fill in the ones you can verify; un-annotated rows are still printed for eyeballing.
    KNOWN_GOLD = {
        '10851': 'George W. Bush',     # Bush attended a Beijing church service, 2008 Olympics
        '10659': 'CEPI',               # Coalition for Epidemic Preparedness Innovations
        '12017': 'Cannes',             # Cannes Film Festival opens ~May 12
        '10645': '161',                # Pentagon released ~16x declassified UFO files
        '10747': '1.5 million',        # Labour's housing pledge
    }

    def _build_q(r):
        try: qt = QuestionType(r.get('qtype', 'mcq'))
        except Exception: qt = QuestionType.MCQ
        return Question(qid=r['qid'], text=r['question_text'], options=r.get('options') or {},
                        qtype=qt, level=r.get('level'), topic=r.get('topic'), language=r.get('language'))

    def _gold_letter(r):
        sub = KNOWN_GOLD.get(str(r['qid']))
        if not sub:
            return None
        for k, v in (r.get('options') or {}).items():
            if sub.lower() in str(v).lower():
                return k
        return None

    print(f"{'qid':>7} | RAG | noRAG | gold | verdict")
    print('-' * 70)
    rag_ok = norag_ok = scored = 0
    for r in rows:
        q = _build_q(r)
        a_rag = pipeline.answer(q).answer          # WITH live retrieval
        a_no  = pipeline_norag.answer(q).answer    # parametric knowledge only
        gold = _gold_letter(r)
        verdict = ''
        if gold:
            scored += 1
            rag_ok   += (a_rag == gold)
            norag_ok += (a_no  == gold)
            verdict = f"RAG {'OK' if a_rag==gold else 'X'} | noRAG {'OK' if a_no==gold else 'X'}"
        print(f"{r['qid']:>7} |  {a_rag}   |  {a_no}    |  {gold or '-'}   | {verdict}")
        print(f"          Q: {r['question_text'][:82]}")

    if scored:
        print(f"\nON {scored} ANNOTATED-GOLD QUESTIONS:   RAG {rag_ok}/{scored}    no-RAG {norag_ok}/{scored}")
    print("\n(Eyeball RAG vs noRAG on the un-annotated rows; add to KNOWN_GOLD to score more.)")

RAG-vs-noRAG ablation is News-only (Maths uses no retrieval) -- skipping for TARGET = math.
